In [1]:
import json

# 读取JSONL文件
results = []
with open('/2022233235/videollm-online/livecc/evaluation/vcrbench/qwen2vl/Qwen2-VL-7B-Instructeval_results.jsonl', 'r') as f:
    for line in f:
        results.append(json.loads(line.strip()))

# 计算is_correct为true的数量
correct_count = sum(1 for result in results if result.get('is_correct', False))

# 计算总数量
total_count = len(results)

# 计算概率
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Total samples: {total_count}")
print(f"Correct predictions: {correct_count}")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


Total samples: 496
Correct predictions: 101
Accuracy: 0.2036 (20.36%)


In [14]:
import json

data = json.load(open('/2022233235/videollm-online/livecc/evaluation/video_augmentation/results/Qwen2-VL-7B-Instruct_continuity_results_stampTrue.json', 'r'))

def evaluate_video_continuity_results( data):
    """
    Evaluate video continuity detection results.
    
    Args:
        predictions: Model predictions (0 for No, 1 for Yes)
        dataset: The dataset used for evaluation
    
    Returns:
        dict: Evaluation metrics
    """
    correct = 0
    total = 0
    continuous_correct = 0
    continuous_total = 0
    discontinuous_correct = 0
    discontinuous_total = 0
    pred_false = 0
    pred_true = 0
    
    for i, datum in enumerate(data):
        total += 1
        predicted_continuous = str(datum['predicted_continuous'])  # 1 for Yes (continuous)
        actual_continuous = str(datum['is_continuous'])
        if predicted_continuous == 'True' :
            pred_true += 1
        if predicted_continuous == 'False' :
            pred_false += 1
        if predicted_continuous == actual_continuous:
            correct += 1
            
        if actual_continuous == 'True':
            continuous_total += 1
            if predicted_continuous == 'True':
                continuous_correct += 1
        else:
            discontinuous_total += 1
            if predicted_continuous == 'False':
                discontinuous_correct += 1
    
    metrics = {
        'overall_accuracy': correct / total if total > 0 else 0,
        'continuous_accuracy': continuous_correct / continuous_total if continuous_total > 0 else 0,
        'discontinuous_accuracy': discontinuous_correct / discontinuous_total if discontinuous_total > 0 else 0,
        'total_samples': total,
        'continuous_samples': continuous_total,
        'discontinuous_samples': discontinuous_total
    }
    
    print(f"Overall Accuracy: {metrics['overall_accuracy']:.4f}")
    print(f"Continuous Videos Accuracy: {metrics['continuous_accuracy']:.4f} ({continuous_correct}/{continuous_total})")
    print(f"Discontinuous Videos Accuracy: {metrics['discontinuous_accuracy']:.4f} ({discontinuous_correct}/{discontinuous_total})")
    print(f"Predicted True: {pred_true}/{total}")
    print(f"Predicted False: {pred_false}/{total}")
    
    return metrics

evaluate_video_continuity_results(data)

Overall Accuracy: 0.4980
Continuous Videos Accuracy: 0.1720 (86/500)
Discontinuous Videos Accuracy: 0.8240 (412/500)
Predicted True: 174/1000
Predicted False: 826/1000


{'overall_accuracy': 0.498,
 'continuous_accuracy': 0.172,
 'discontinuous_accuracy': 0.824,
 'total_samples': 1000,
 'continuous_samples': 500,
 'discontinuous_samples': 500}

In [2]:
import json

# 读取COF数据集的JSONL文件
ovobench_data = []
with open('/2022233235/videollm-online/livecc/evaluation/ovobench/results/Qwen2-VL-7B-Instruct2.jsonl', 'r') as f:
    for line in f:
        ovobench_data.append(json.loads(line.strip()))

print(f"Loaded {len(ovobench_data)} samples from ovobench data.")


ovobench_data2 = json.load(open('/2022233235/videollm-online/livecc/evaluation/ovobench/results/Qwen2-VL-7B-Instruct.json', 'r'))
print(f"Loaded {len(ovobench_data2)} samples from ovobench data.")


# 根据id，找到在data2中对的在data中不对的结果，print出来

# 构建data和data2的id到样本的映射
data2_by_id = {item['id']: item for item in ovobench_data}
data1_by_id = {item['id']: item for item in ovobench_data2}

# 遍历data2，找出data2中预测正确但data1中预测错误的样本
c = 0
for id_, item2 in data2_by_id.items():
    item1 = data1_by_id.get(id_)
    if item1 is None:
        continue
    # 判断data2是否预测正确
    gt2 = item2.get('answer', None)
    pred2 = item2.get('response', None)
    correct2 = item2.get('is_correct', None)
    if correct2 is None:
        correct2 = (str(gt2).strip().lower() == str(pred2).strip().lower())
    # 判断data1是否预测错误
    gt1 = item1.get('answer', None)
    pred1 = item1.get('response', None)
    correct1 = item1.get('is_correct', None)
    if correct1 is None:
        correct1 = (str(gt1).strip().lower() == str(pred1).strip().lower())
    if correct2 and not correct1:
        print(f"id: {id_}")
        print("data2:", json.dumps(item2, ensure_ascii=False, indent=2))
        print("data1:", json.dumps(item1, ensure_ascii=False, indent=2))
        print("="*40)
        c += 1
print(c)    







Loaded 1468 samples from ovobench data.
Loaded 1468 samples from ovobench data.
id: 1365
data2: {
  "id": 1365,
  "task": "OCR",
  "question": "What text is displayed on the Computer screen now?",
  "options": [
    "A. A healthy life starts with GOOD",
    "B. A healthy life starts with HOOD",
    "C. A healthy life starts with FOOD",
    "D. A healthy life starts with FOODIE"
  ],
  "answer": "C",
  "response": "C",
  "generated_text": "C. A healthy life starts with FOOD",
  "is_correct": true
}
data1: {
  "id": 1365,
  "task": "OCR",
  "question": "What text is displayed on the Computer screen now?",
  "options": [
    "A. A healthy life starts with GOOD",
    "B. A healthy life starts with HOOD",
    "C. A healthy life starts with FOOD",
    "D. A healthy life starts with FOODIE"
  ],
  "answer": "C",
  "response": "A"
}
id: 72
data2: {
  "id": 72,
  "task": "EPM",
  "question": "What color is the towel I wiped hands with?",
  "options": [
    "A. white",
    "B. pink",
    "C. blu